<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/05-una-sola-prueba-puede-enganar/ejercicio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Ejercicio 2.5 · Una sola prueba puede engañar

**Serie:** Del Excel al Machine Learning · **Temporada 2** · 🎬 Video próximamente

Antes de empezar, guarda tu copia: **Archivo → Guardar una copia en Drive**.

Escribe tu respuesta en cada celda de código y ejecútala con **Shift + Enter**.

### Prepara los datos

Ejecuta estas celdas. Dejan listas `serie`, `FERIADOS`, `meses` y `ventanas`. `ventanas` debe tener `10` filas.

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [ ]:
FERIADOS = pd.to_datetime(
    [
        "2024-01-01",
        "2024-03-28",
        "2024-03-29",
        "2024-05-01",
        "2024-06-29",
        "2024-07-28",
        "2024-07-29",
        "2024-08-30",
        "2024-10-08",
        "2024-11-01",
        "2024-12-08",
        "2024-12-09",
        "2024-12-25",
        "2025-01-01",
        "2025-04-17",
        "2025-04-18",
        "2025-05-01",
        "2025-06-29",
        "2025-07-28",
        "2025-07-29",
        "2025-08-30",
        "2025-10-08",
        "2025-11-01",
        "2025-12-08",
        "2025-12-09",
        "2025-12-25",
    ]
)

In [ ]:
def calendario(dias):
    """Una columna por día de la semana y por mes, más la tendencia: los días desde el inicio."""
    x = pd.DataFrame(index=dias)
    for dia in range(7):
        x[f"dia_{dia}"] = (dias.dayofweek == dia).astype(int)
    for mes in range(1, 13):
        x[f"mes_{mes}"] = (dias.month == mes).astype(int)
    x["tendencia"] = (dias - pd.Timestamp("2024-01-01")).days
    return x

In [ ]:
def tres_pronosticos(serie, feriados, mes):
    """Pronostica cada día de un mes con tres métodos, usando solo lo anterior a ese mes."""
    inicio = pd.Timestamp(mes + "-01")
    dias = pd.date_range(inicio, inicio + pd.offsets.MonthEnd(0))
    antes = serie[serie.index < inicio]
    laborables = antes[~antes.index.isin(feriados)]

    # Promedio por día de la semana de las últimas 8 semanas.
    ultimas = laborables[laborables.index >= inicio - pd.Timedelta(days=56)]
    por_dia = ultimas.groupby(ultimas.index.dayofweek).mean()
    ocho_semanas = pd.Series(dias.dayofweek.map(por_dia), index=dias)

    # Índice: cuánto subió ese mes en 2024 frente a las 8 semanas previas de 2024.
    hace_un_anio = inicio - pd.Timedelta(days=364)
    base = laborables[
        (laborables.index >= hace_un_anio - pd.Timedelta(days=56))
        & (laborables.index < hace_un_anio)
    ]
    mismo_mes = laborables[
        (laborables.index.year == 2024) & (laborables.index.month == inicio.month)
    ]
    regla = ocho_semanas * mismo_mes.mean() / base.mean()

    modelo = LinearRegression().fit(calendario(laborables.index), laborables)
    regresion = pd.Series(modelo.predict(calendario(dias)), index=dias)

    # Todos los métodos ponen los feriados en cero: esos días no se vende.
    tabla = pd.DataFrame(
        {"real": serie[dias], "ocho_semanas": ocho_semanas, "regla": regla, "regresion": regresion}
    )
    metodos = ["ocho_semanas", "regla", "regresion"]
    tabla.loc[tabla.index.isin(feriados), metodos] = 0
    return tabla

In [ ]:
def probar_en_ventanas(serie, feriados, meses):
    """Pronostica cada mes con lo anterior a él, y devuelve el error de cada método."""
    filas = []
    for mes in meses:
        tabla = tres_pronosticos(serie, feriados, mes)
        fila = {"mes": mes}
        for metodo in ["ocho_semanas", "regla", "regresion"]:
            fila[metodo] = (tabla["real"] - tabla[metodo]).abs().mean()
            fila[metodo + "_mes"] = abs(tabla["real"].sum() - tabla[metodo].sum())
        filas.append(fila)
    return pd.DataFrame(filas).set_index("mes").round(0)

In [ ]:
ventas = pd.read_csv(DATOS + "ventas.csv", parse_dates=["fecha"])

serie = ventas.groupby("fecha")["total"].sum().asfreq("D", fill_value=0)

meses = [f"2025-{numero:02d}" for numero in range(3, 13)]

ventanas = probar_en_ventanas(serie, FERIADOS, meses)

print(len(ventanas))

### 1. El peor y el mejor mes

¿En qué mes se equivoca más `regla` por día, y en cuál menos? Usa `.idxmax()` y `.idxmin()`. El peor debe ser `2025-07`.

In [ ]:
# Escribe tu código aquí

### 2. ¿Cuántas veces gana?

Cuenta en cuántos meses `regla` tiene menos error por día que `ocho_semanas`. Deben ser `7` de 10.

In [ ]:
# Escribe tu código aquí

### 3. Por el total del mes

Cuenta en cuántos meses `regresion_mes` es menor que `regla_mes`. Calcula también la media de cada una. ¿Sale lo mismo que por día?

In [ ]:
# Escribe tu código aquí

### 4. La mentira de un solo mes

Encuentra el mes en que la regla le saca más ventaja a la regresión en `_mes`, y el mes en que pasa al revés. Calcula `ventanas["regresion_mes"] - ventanas["regla_mes"]`, y usa `.idxmax()` y `.idxmin()`.

In [ ]:
# Escribe tu código aquí

### 5. Otras ventanas

Calcula `probar_en_ventanas` solo para julio a diciembre, y guárdala en `segundo_semestre`. ¿Cuál es el error medio por día de `regla`? Debe dar `2,935`.

In [ ]:
# Escribe tu código aquí

### 6. Tu conclusión

Responde en un comentario. Escribe dos frases para tu jefe: qué método recomiendas y por qué. Di también qué **no** se puede afirmar con estos datos.

In [ ]:
# Escribe tu código aquí

## 🎉 ¡Terminaste!

Compara tus respuestas con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/05-una-sola-prueba-puede-enganar/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0